# LangChain JSON Output Parser Reference

Developer-facing statements defined in `langchain_core.output_parsers.json`.

# `PydanticBaseModel`

Type alias covering both Pydantic v1 and v2 base-model classes.

```python
PydanticBaseModel = BaseModel | pydantic.BaseModel
```

---

# `JsonOutputParser: BaseCumulativeTransformOutputParser[Any]`

Parses language-model output into JSON objects.

In streaming mode, it can emit partial JSON objects containing the keys received so far. When the inherited `diff` option is enabled, it emits JSON Patch operations describing changes between successive objects.

## Fields

```python
pydantic_object: Annotated[type[TBaseModel] | None, SkipValidation()] = None # Pydantic model class whose schema is used in format instructions
```

In this pinned implementation, `pydantic_object` is used to generate schema-based format instructions. `parse_result()` does not validate parsed output against the model.

## Constructor

```python
JsonOutputParser(
    *,
    pydantic_object: Annotated[type[TBaseModel] | None, SkipValidation()] = None, # Optional Pydantic model class
) -> None
```

## Methods

### `parse_result`

Parses the first candidate generation as JSON.

```python
@override
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: bool = False, # Whether to tolerate incomplete JSON
) -> Any # Parsed JSON value, or None for an invalid partial result
```

The first generation's text is stripped and parsed with `parse_json_markdown()`.

When `partial=True`, a `JSONDecodeError` produces `None`. Otherwise, invalid JSON raises `OutputParserException` with the stripped text stored as `llm_output`.

### `parse`

Parses one text value as JSON.

```python
parse(
    self,
    text: str, # Language-model output to parse
) -> Any # Parsed JSON value
```

The method wraps `text` in a `Generation` and delegates to `parse_result()`.

### `get_format_instructions`

Returns instructions describing the required JSON output format.

```python
get_format_instructions(
    self,
) -> str # JSON output-format instructions
```

When `pydantic_object` is `None`, the method returns `"Return a JSON object."`.

When a Pydantic model is supplied, the method obtains its JSON schema, copies it, removes top-level `"title"` and `"type"` fields when present, serializes the reduced schema with non-ASCII characters preserved, and inserts it into `JSON_FORMAT_INSTRUCTIONS`.

## Behaviour

For streaming differences, successive parsed objects are compared with `jsonpatch.make_patch()`, and the emitted value is the resulting patch-operation list.

---

# `SimpleJsonOutputParser`

Backwards-compatible alias for `JsonOutputParser`.

```python
SimpleJsonOutputParser = JsonOutputParser
```

In [ ]:
from pydantic import BaseModel # Import BaseModel for defining a JSON schema

from langchain_core.exceptions import OutputParserException # Import the real parser exception
from langchain_core.messages import AIMessage # Import a real LangChain message
from langchain_core.output_parsers import JsonOutputParser # Import the real JSON parser
from langchain_core.outputs import Generation # Import Generation for parse_result


class Person(BaseModel): # Define the expected JSON structure
    name: str # Require a name field
    age: int # Require an age field


parser = JsonOutputParser(pydantic_object=Person) # Create the parser with schema instructions

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display schema-based instructions

json_text = '{"name": "Saad", "age": 22}' # Create valid JSON text

parsed_data = parser.parse(json_text) # Parse the JSON string
print("\nParsed with parse():", parsed_data) # Display the parsed dictionary

invoked_data = parser.invoke(json_text) # Parse through the runnable interface
print("Parsed with invoke():", invoked_data) # Display the runnable result

message = AIMessage(content='{"name": "Aman", "age": 25}') # Create a message containing JSON
message_data = parser.invoke(message) # Parse the AIMessage
print("Parsed AIMessage:", message_data) # Display the parsed message content

partial_generation = Generation( # Create an incomplete streaming result
    text='{"name": "Saad", "age": 22' # Omit the final closing brace
)

partial_data = parser.parse_result( # Parse incomplete JSON
    [partial_generation], # Provide the generation list
    partial=True, # Enable partial parsing
)

print("Parsed partial JSON:", partial_data) # Display the recovered object

markdown_json = """```json
{"language": "Python", "level": "beginner"}
```""" # Create JSON inside a Markdown code block

markdown_data = parser.parse(markdown_json) # Parse JSON from Markdown
print("Parsed Markdown JSON:", markdown_data) # Display the parsed object

try: # Start invalid JSON handling
    parser.parse('{"name": "Saad", age: 22}') # Parse malformed JSON
except OutputParserException as error: # Catch the real LangChain exception
    print("\nParser error:", error) # Display the error
    print("Invalid model output:", error.llm_output) # Display the original output